# 06 - Judge contest v2 for the generated pairs

**Why this notebook.** The first judge contest (notebook 05) failed: every judge answered SARCASTIC to almost every headline, so none reached 0.75 balanced accuracy and the pairs were kept unfiltered. The judges still leaned the right way, so the problem was a bias, not blindness. This notebook fixes the bias in two ways:
- **A score instead of a word.** For each headline we read how likely the model finds the answer `SARCASTIC` and how likely `SERIOUS`, and take the difference (a log-probability difference). Then we choose the cut-off (threshold) on real headlines. A model that leans towards SARCASTIC simply gets a higher cut-off.
- **A better question (prompt v2).** It says that every headline, real or satirical, is lowercase without punctuation, and shows 3 real labelled examples of each class. Prompt v1 is the old question, kept for comparison.

**Candidates** (9): SmolLM3-3B, Qwen2.5-7B, Gemma-2-9B and Mistral-7B, each with prompt v1 and v2, plus an off-the-shelf irony classifier trained on tweets (`cardiffnlp/twitter-roberta-base-irony`), which never saw our data. A sarcasm model trained on the News Headlines dataset is not allowed: it may have seen our test headlines.

**Rules that keep it honest (fixed before the run):**
1. Judges are scored only on **real, labelled training headlines**: never on the generated pairs, never on dev or test.
2. Two separate sets of 400 real headlines (200 per class). The threshold is chosen on the **tune** set; the score that decides pass or fail is measured on the **report** set.
3. A judge passes if its report-set balanced accuracy is at least **0.75** (same bar as before).
4. A generator model never judges pairs it wrote itself. For each author model, the passing candidate with the best tune-set score judges its pairs. If no candidate passes for an author, its pairs stay unfiltered and are flagged.
5. A pair is kept if its judge calls the sarcastic side sarcastic **and** the serious side serious.

**What it writes** (`/kaggle/working/judge_v2/`, zipped as `judge_v2.zip`): `judge_v2_contest.json` (scores of all 9 candidates, the chosen judges), `real_scores.csv`, `gen_scores.csv`, `pairs_judged.csv` (one row per pair), `pairs_train_filtered.csv` (same columns as `pairs_train.csv`) and `manifest_judge_v2.json` (SHA-256 of every file).

The unfiltered `pairs_train.csv` stays the **main** Pass 4 dataset. The filtered set is an ablation: does filtering the pairs matter?

**How to run:** right-hand panel: GPU T4 x2, Internet on, secret `HF_TOKEN`; inputs `gainsplit-processed` **and** the output of the finalize run of notebook 05 (it holds `pairgen/pairs_train.csv` and `pairgen/manifest_pairs.json`). Then **Save Version > Save & Run All (Commit)**. Expected time: about 1-4 hours (contest about 30-40 minutes; scoring the generated headlines depends on which judges win). Download `judge_v2.zip` from the version's Output.

## 1. Get the code and run the tests

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_judge.py", "tests/test_pairgen.py", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 2. Settings (no need to change anything)
`batch` is how many headlines go through a model at once; Gemma gets a smaller batch because it nearly fills the two GPUs. `SEED` picks the real headlines; it differs from the generation seed on purpose (a fresh draw).

In [ ]:
from pathlib import Path
LLMS = {
    "smollm3": {"id": "HuggingFaceTB/SmolLM3-3B",           "extra": {},                                  "batch": 16},
    "qwen":    {"id": "Qwen/Qwen2.5-7B-Instruct",           "extra": {},                                  "batch": 16},
    "gemma":   {"id": "google/gemma-2-9b-it",               "extra": {"attn_implementation": "eager"}, "batch": 8},
    "mistral": {"id": "mistralai/Mistral-7B-Instruct-v0.3", "extra": {},                                  "batch": 16},
}
IRONY = {"name": "irony", "id": "cardiffnlp/twitter-roberta-base-irony", "batch": 64}
GENERATORS = ["qwen", "gemma", "mistral"]   # the models that wrote the pairs
N_TUNE, N_REPORT, N_SHOT = 400, 400, 3      # real headlines; N_SHOT examples per class in prompt v2
SEED = 2027
JUDGE_MIN = 0.75
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/judge_v2")
OUT.mkdir(parents=True, exist_ok=True)

## 3. Data: real headlines and the pair dataset
Reads the training split, then finds `pairs_train.csv` in the inputs and checks its SHA-256 fingerprint against `manifest_pairs.json` from the finalize run, so we judge exactly the frozen dataset. If the file is found in more than one input (for example in the notebook output and in a new version of `gainsplit-processed`), all copies must be identical.

Then it draws the real headlines: 3 + 3 few-shot examples (shown in prompt v2, never scored), 400 for the tune set and 400 for the report set, all from the training split and with no overlap.

In [ ]:
import json, hashlib
import numpy as np
import pandas as pd
from gainsplit.data import find_data_dir
from gainsplit import judge as jd

sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
py = lambda o: o.item() if hasattr(o, "item") else str(o)   # numpy numbers -> plain numbers in JSON
DATA_DIR = find_data_dir(str(INPUT))
train = pd.read_csv(DATA_DIR / "headlines_train.csv", keep_default_na=False)

found = sorted(INPUT.rglob("pairs_train.csv"))
assert found, "pairs_train.csv not found: add the output of the notebook 05 finalize run as input"
hashes = {str(p): sha(p) for p in found}
assert len(set(hashes.values())) == 1, f"different copies of pairs_train.csv: {hashes}"
PAIRS_FILE = found[0]
PAIRS_SHA = hashes[str(PAIRS_FILE)]
mf = [p.parent / "manifest_pairs.json" for p in found if (p.parent / "manifest_pairs.json").exists()]
assert mf, "manifest_pairs.json not found next to pairs_train.csv"
PAIRS_MANIFEST = json.loads(mf[0].read_text())
assert PAIRS_MANIFEST["sha256"]["pairs_train.csv"] == PAIRS_SHA, "pairs_train.csv does not match its manifest"
rows = pd.read_csv(PAIRS_FILE, keep_default_na=False)
print("pairs file:", PAIRS_FILE, "| rows:", len(rows), "| pairs:", rows["seed_id"].nunique(), "| SHA-256 matches the manifest")
print(rows.groupby(["model", "label"]).size().unstack())

shots, tune, report = jd.pick_real_sets(train, N_TUNE, N_REPORT, N_SHOT, SEED)
real = pd.concat([tune.assign(set="tune"), report.assign(set="report")], ignore_index=True)
print("few-shot examples:", len(shots), "| tune:", len(tune), "| report:", len(report))
print("\nprompt v2 for the first tune headline:\n")
print(jd.build_prompt(tune["text"].iloc[0], "v2", shots))

## 4. Scoring functions (GPU)
**Language models.** For every headline the prompt is put into the model's chat format, then the answer word is appended, and one pass through the model gives the log-probability of that answer. We do this for `SARCASTIC` and for `SERIOUS`; the score is the difference. Higher score = the model finds SARCASTIC more likely. No text is generated, so this is fast and the same headline always gets the same score.

Technical notes: sequences are padded on the right (so every headline keeps its normal token positions) and sorted by length (less padding). To save memory the full output layer is applied only at the answer positions, not to every token (Gemma's vocabulary has 256,000 entries). Gemma 2 squashes its outputs with a "soft cap"; the same is applied here so the numbers match the model's own.

**Irony classifier.** A small RoBERTa model; its score is the irony logit minus the non-irony logit.

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
HF_TOKEN = get_secret("HF_TOKEN")
print("HF_TOKEN found:", HF_TOKEN is not None, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def load_lm(name):
    spec = LLMS[name]
    tok = AutoTokenizer.from_pretrained(spec["id"], token=HF_TOKEN)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(spec["id"], token=HF_TOKEN, torch_dtype=torch.float16,
                                                 device_map="auto", **spec["extra"])
    model.eval()
    return tok, model

def free():
    """Release GPU memory after `del model, tok` (the caller must drop its own references first)."""
    gc.collect(); torch.cuda.empty_cache()

def _answer_logprob(tok, model, prompt_ids, answer, batch):
    a_ids = tok(answer, add_special_tokens=False)["input_ids"]
    k = len(a_ids)
    seqs = [p + a_ids for p in prompt_ids]
    order = np.argsort([len(s) for s in seqs], kind="stable")
    out = np.zeros(len(seqs))
    body, head = model.model, model.get_output_embeddings()
    cap = getattr(model.config, "final_logit_softcapping", None)
    dev = model.get_input_embeddings().weight.device
    for i in range(0, len(seqs), batch):
        idx = order[i:i + batch]
        part = [seqs[j] for j in idx]
        T, B = max(len(s) for s in part), len(part)
        ids = torch.full((B, T), tok.pad_token_id, dtype=torch.long)
        mask = torch.zeros((B, T), dtype=torch.long)
        for r, s in enumerate(part):
            ids[r, :len(s)] = torch.tensor(s); mask[r, :len(s)] = 1
        with torch.no_grad():
            h = body(input_ids=ids.to(dev), attention_mask=mask.to(dev)).last_hidden_state
            lens = torch.tensor([len(s) for s in part])
            pos = (lens - k - 1).unsqueeze(1) + torch.arange(k)          # positions that predict the answer tokens
            hs = h[torch.arange(B).unsqueeze(1).to(h.device), pos.to(h.device)]
            logits = head(hs.to(head.weight.device)).float()
            if cap:
                logits = torch.tanh(logits / cap) * cap
            lp = torch.log_softmax(logits, dim=-1)
            tgt = torch.tensor(a_ids, device=lp.device).expand(B, k).unsqueeze(-1)
            out[idx] = lp.gather(-1, tgt).squeeze(-1).sum(-1).cpu().numpy()
    return out

def lm_scores(tok, model, name, prompts):
    """log P(SARCASTIC) - log P(SERIOUS) for each prompt."""
    ids = [tok(tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False), add_special_tokens=False)["input_ids"] for p in prompts]
    b = LLMS[name]["batch"]
    return _answer_logprob(tok, model, ids, jd.ANSWERS[1], b) - _answer_logprob(tok, model, ids, jd.ANSWERS[0], b)

def irony_scores(texts):
    tok = AutoTokenizer.from_pretrained(IRONY["id"])
    model = AutoModelForSequenceClassification.from_pretrained(IRONY["id"]).to("cuda").eval()
    names = {k: v.lower() for k, v in model.config.id2label.items()}
    pos = [k for k, v in names.items() if v == "irony"]
    neg = [k for k, v in names.items() if v in ("non_irony", "not_irony", "non-irony")]
    pos, neg = (pos[0], neg[0]) if pos and neg else (1, 0)
    print("irony labels:", names, "| irony index:", pos)
    out = []
    for i in range(0, len(texts), IRONY["batch"]):
        enc = tok(list(texts[i:i + IRONY["batch"]]), return_tensors="pt", padding=True, truncation=True, max_length=64).to("cuda")
        with torch.no_grad():
            lg = model(**enc).logits.float()
        out += (lg[:, pos] - lg[:, neg]).cpu().tolist()
    del model, tok
    free()
    return np.array(out)

## 5. The contest on real headlines (about 30-40 minutes)
Every candidate scores the 800 real headlines (tune + report). The threshold is set on the tune set (the cut-off with the best balanced accuracy) and then applied unchanged to the report set. The table shows, for every candidate:
- `tune_bacc` / `report_bacc`: balanced accuracy (mean of the accuracy on sarcastic and on serious headlines; 0.5 = coin flip). **`report_bacc` decides pass or fail.**
- `report_auc`: how well the scores rank sarcastic above serious, whatever the threshold (0.5 = no ability, 1 = perfect).
- `report_sarcastic_rate`: share of report headlines called sarcastic (about 0.5 is healthy; the old judges were near 1.0).

In [ ]:
real_scores = []
for name in LLMS:
    t0 = time.time()
    tok, model = load_lm(name)
    for v in jd.PROMPT_VERSIONS:
        s = lm_scores(tok, model, name, [jd.build_prompt(t, v, shots) for t in real["text"]])
        real_scores.append(real.assign(candidate=f"{name}|{v}", base=name, score=s))
    del model, tok; free()
    print(f"{name}: {(time.time() - t0) / 60:.1f} min")
t0 = time.time()
real_scores.append(real.assign(candidate=IRONY["name"], base=IRONY["name"], score=irony_scores(real["text"].tolist())))
print(f"irony: {(time.time() - t0) / 60:.1f} min")
real_scores = pd.concat(real_scores, ignore_index=True)
real_scores.to_csv(OUT / "real_scores.csv", index=False)

results = {}
for cand, g in real_scores.groupby("candidate", sort=False):
    tu, rp = g[g["set"] == "tune"], g[g["set"] == "report"]
    results[cand] = {"base": g["base"].iloc[0], **jd.evaluate(tu["label"], tu["score"], rp["label"], rp["score"])}
table = pd.DataFrame(results).T.sort_values("tune_bacc", ascending=False)
table["passes"] = table["report_bacc"] >= JUDGE_MIN
display(table)

JUDGES = jd.choose_judges(results, GENERATORS, JUDGE_MIN)
print("judge per author model:", JUDGES)
if not any(JUDGES.values()):
    print(f"No candidate reached {JUDGE_MIN} on the report set: no filter; the pairs stay unfiltered.")
contest = {"seed": SEED, "n_tune": N_TUNE, "n_report": N_REPORT, "n_shot_per_class": N_SHOT, "min_score": JUDGE_MIN,
           "shots": shots.to_dict("records"), "candidates": {**{n: s["id"] for n, s in LLMS.items()}, IRONY["name"]: IRONY["id"]},
           "results": results, "judges": JUDGES,
           "prompt_v2_example": jd.build_prompt("<headline>", "v2", shots)}
(OUT / "judge_v2_contest.json").write_text(json.dumps(contest, indent=2, default=py))

## 6. Score the generated headlines with the chosen judges, then filter
Only the judges that were chosen score generated headlines, and each one only the pairs of the authors it judges (never its own). If no judge was chosen, nothing is scored and `pairs_train_filtered.csv` equals the unfiltered file. The last tables show, per author model, how many pairs were judged and kept, and how often each judge called each side sarcastic: a large gap between the two sides means the pairs differ in the intended direction.

In [ ]:
gen_scores = []
for cand in sorted({j for j in JUDGES.values() if j}):
    authors = [a for a, j in JUDGES.items() if j == cand]
    items = rows[rows["model"].isin(authors)]
    t0 = time.time()
    if cand == IRONY["name"]:
        s = irony_scores(items["text"].tolist())
    else:
        base, v = cand.split("|")
        tok, model = load_lm(base)
        s = lm_scores(tok, model, base, [jd.build_prompt(t, v, shots) for t in items["text"]])
        del model, tok; free()
    gen_scores.append(pd.DataFrame({"id": items["id"].values, "candidate": cand, "score": s}))
    print(f"{cand}: {len(items)} generated headlines ({', '.join(authors)}) in {(time.time() - t0) / 60:.1f} min")
gen_scores = (pd.concat(gen_scores, ignore_index=True) if gen_scores
              else pd.DataFrame(columns=["id", "candidate", "score"]))
gen_scores.to_csv(OUT / "gen_scores.csv", index=False)

filtered, per_pair = jd.filter_pairs(rows, gen_scores, JUDGES, results)
per_pair.to_csv(OUT / "pairs_judged.csv", index=False)
filtered.to_csv(OUT / "pairs_train_filtered.csv", index=False)
display(per_pair.groupby("model").agg(pairs=("kept", "size"), judged=("judged", "mean"), kept=("kept", "mean"),
                                      sarcastic_side_ok=("sarcastic_ok", "mean"), serious_side_ok=("serious_ok", "mean")).round(3))
if len(gen_scores):
    g = gen_scores.merge(rows[["id", "model", "label"]], on="id")
    g["called_sarcastic"] = g["score"] >= g["candidate"].map(lambda c: results[c]["threshold"])
    display(g.pivot_table(index=["candidate", "model"], columns="label", values="called_sarcastic", aggfunc="mean")
             .rename(columns={1: "sarcastic side", 0: "serious side"}).round(3))
print("pairs kept:", per_pair["kept"].sum(), "of", len(per_pair), "| training rows:", len(filtered))

## 7. Manifest and download

In [ ]:
import shutil
files = ["judge_v2_contest.json", "real_scores.csv", "gen_scores.csv", "pairs_judged.csv", "pairs_train_filtered.csv"]
manifest = {
    "commit": COMMIT, "input_pairs_train_sha256": PAIRS_SHA, "input_pairs_commit": PAIRS_MANIFEST.get("commit"),
    "judges": JUDGES, "min_score": JUDGE_MIN,
    "scores": {c: {k: r[k] for k in ("threshold", "tune_bacc", "report_bacc", "report_auc", "report_sarcastic_rate")}
               for c, r in results.items()},
    "counts": {"pairs": int(len(per_pair)), "pairs_judged": int(per_pair["judged"].sum()),
               "pairs_kept": int(per_pair["kept"].sum()), "training_rows": int(len(filtered)),
               "kept_by_model": per_pair.groupby("model")["kept"].sum().astype(int).to_dict()},
    "hours": round((time.time() - T_START) / 3600, 2),
    "sha256": {f: sha(OUT / f) for f in files},
}
(OUT / "manifest_judge_v2.json").write_text(json.dumps(manifest, indent=2, default=py))
print(json.dumps({k: v for k, v in manifest.items() if k != "sha256"}, indent=2, default=py))
print(shutil.make_archive("/kaggle/working/judge_v2", "zip", OUT))